[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/vision_transformers/blob/main/autoencoder_vae.ipynb)

In [ ]:
import os
import json
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(device)
print(f'Using device: {device} ({gpu_name})')
RESULTS_DIR = 'autoencoder_vae'
os.makedirs(RESULTS_DIR, exist_ok=True)
DATASETS_TO_RUN = ['cifar10', 'eurosat']
BATCH_SIZE = 128
LATENT_DIM = 128
AE_EPOCHS = 15
VAE_EPOCHS = 20
LR = 1e-3


In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


DATASET_CONFIGS = {
    "cifar10": {"in_channels": 3, "image_size": 32},
    "eurosat": {"in_channels": 3, "image_size": 64},
    "stl10": {"in_channels": 3, "image_size": 64},
    "celeba": {"in_channels": 3, "image_size": 64},
    "mnist": {"in_channels": 1, "image_size": 32},
}


def get_dataloaders(dataset_name="cifar10", batch_size=128, data_dir="./data", num_workers=2):
    """Load train and test DataLoaders for CIFAR-10, EuroSAT, STL-10, CelebA, or MNIST."""
    name = dataset_name.lower()
    if name not in DATASET_CONFIGS:
        raise ValueError(f"Unsupported dataset '{dataset_name}'. Choose from {list(DATASET_CONFIGS.keys())}.")

    cfg = DATASET_CONFIGS[name]
    image_size = cfg["image_size"]

    if name == "celeba":
        transform = transforms.Compose([
            transforms.CenterCrop(178),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
    else:
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])

    if name == "cifar10":
        train_dataset = datasets.CIFAR10(root=data_dir, train=True, transform=transform, download=True)
        test_dataset = datasets.CIFAR10(root=data_dir, train=False, transform=transform, download=True)
    elif name == "eurosat":
        full_dataset = datasets.EuroSAT(root=data_dir, transform=transform, download=True)
        train_size = int(0.8 * len(full_dataset))
        test_size = len(full_dataset) - train_size
        generator = torch.Generator().manual_seed(42)
        train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size], generator=generator)
    elif name == "stl10":
        train_dataset = datasets.STL10(root=data_dir, split="train+unlabeled", transform=transform, download=True)
        test_dataset = datasets.STL10(root=data_dir, split="test", transform=transform, download=True)
    elif name == "celeba":
        train_dataset = datasets.CelebA(root=data_dir, split="train", transform=transform, download=True)
        test_dataset = datasets.CelebA(root=data_dir, split="test", transform=transform, download=True)
    else:
        train_dataset = datasets.MNIST(root=data_dir, train=True, transform=transform, download=True)
        test_dataset = datasets.MNIST(root=data_dir, train=False, transform=transform, download=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader, cfg


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvAutoencoder(nn.Module):
    """Convolutional Autoencoder supporting RGB/grayscale images (32x32 or 64x64)."""

    def __init__(self, in_channels=3, image_size=32, latent_dim=128, base_channels=64):
        super().__init__()
        self.in_channels = in_channels
        self.image_size = image_size
        self.latent_dim = latent_dim
        self.feat_size = image_size // 8
        self.flat_dim = base_channels * 4 * self.feat_size * self.feat_size

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.fc_enc = nn.Linear(self.flat_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, self.flat_dim)
        self.base_channels = base_channels

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(base_channels, in_channels, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_enc(h.flatten(1))

    def decode(self, z):
        h = self.fc_dec(z).view(-1, self.base_channels * 4, self.feat_size, self.feat_size)
        return self.decoder(h)

    def forward(self, x):
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z


class ConvEncoder(nn.Module):
    """Convolutional VAE Encoder mapping images to (mu, logvar)."""

    def __init__(self, in_channels=3, image_size=32, latent_dim=128, base_channels=64):
        super().__init__()
        self.feat_size = image_size // 8
        self.flat_dim = base_channels * 4 * self.feat_size * self.feat_size

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.fc_mu = nn.Linear(self.flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim, latent_dim)

    def forward(self, x):
        h = self.conv(x).flatten(1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class ConvDecoder(nn.Module):
    """Convolutional VAE Decoder reconstructing images from latent vector z."""

    def __init__(self, in_channels=3, image_size=32, latent_dim=128, base_channels=64):
        super().__init__()
        self.base_channels = base_channels
        self.feat_size = image_size // 8
        self.flat_dim = base_channels * 4 * self.feat_size * self.feat_size

        self.fc = nn.Linear(latent_dim, self.flat_dim)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(base_channels, in_channels, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        h = self.fc(z).view(-1, self.base_channels * 4, self.feat_size, self.feat_size)
        return self.deconv(h)


class ConvVAE(nn.Module):
    """Convolutional Variational Autoencoder with reparameterization trick."""

    def __init__(self, in_channels=3, image_size=32, latent_dim=128, base_channels=64):
        super().__init__()
        self.in_channels = in_channels
        self.image_size = image_size
        self.latent_dim = latent_dim
        self.encoder = ConvEncoder(in_channels, image_size, latent_dim, base_channels)
        self.decoder = ConvDecoder(in_channels, image_size, latent_dim, base_channels)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decoder(z)
        return x_recon, mu, logvar


def vae_loss_function(recon_x, x, mu, logvar, kld_weight=1.0):
    """ELBO loss = pixel-wise BCE + weighted KL divergence."""
    bce = F.binary_cross_entropy(recon_x, x, reduction="sum")
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kld_weight * kld, bce, kld


In [ ]:
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim



def train_autoencoder(
    model,
    train_loader,
    epochs=15,
    lr=1e-3,
    device="cpu",
    checkpoint_path="autoencoder.pth",
):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        total_loss = 0.0
        for images, _ in train_loader:
            images = images.to(device)
            recon_images, _ = model(images)
            loss = criterion(recon_images, images)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)

        avg_loss = total_loss / len(train_loader.dataset)
        epoch_time = time.time() - t0
        history.append({
            "epoch": epoch + 1,
            "mse_loss": round(avg_loss, 6),
            "epoch_time_sec": round(epoch_time, 2),
        })
        print(f"[Autoencoder] Epoch [{epoch + 1}/{epochs}] - MSE Loss: {avg_loss:.6f} ({epoch_time:.2f}s)")

    if checkpoint_path:
        torch.save(model.state_dict(), checkpoint_path)

    return model, history


def train_vae(
    model,
    train_loader,
    epochs=20,
    lr=1e-3,
    kld_weight=1.0,
    device="cpu",
    checkpoint_path="vae.pth",
):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        total_loss = 0.0
        total_bce = 0.0
        total_kld = 0.0

        for images, _ in train_loader:
            images = images.to(device)
            recon_images, mu, logvar = model(images)
            loss, bce, kld = vae_loss_function(recon_images, images, mu, logvar, kld_weight=kld_weight)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_bce += bce.item()
            total_kld += kld.item()

        n_samples = len(train_loader.dataset)
        avg_total = total_loss / n_samples
        avg_bce = total_bce / n_samples
        avg_kld = total_kld / n_samples
        epoch_time = time.time() - t0
        history.append({
            "epoch": epoch + 1,
            "total_loss": round(avg_total, 4),
            "bce_loss": round(avg_bce, 4),
            "kld_loss": round(avg_kld, 4),
            "epoch_time_sec": round(epoch_time, 2),
        })
        print(
            f"[VAE] Epoch [{epoch + 1}/{epochs}] - "
            f"Total Loss: {avg_total:.4f} | "
            f"BCE: {avg_bce:.4f} | "
            f"KLD: {avg_kld:.4f} ({epoch_time:.2f}s)"
        )

    if checkpoint_path:
        torch.save(model.state_dict(), checkpoint_path)

    return model, history


def save_training_results(results_dict, save_path="training_results.json"):
    with open(save_path, "w") as f:
        json.dump(results_dict, f, indent=2)
    return results_dict


In [ ]:
import torch
import matplotlib.pyplot as plt


def _to_display_image(tensor):
    img = tensor.cpu()
    if img.shape[0] == 1:
        return img.squeeze(0), "gray"
    return img.permute(1, 2, 0).clamp(0.0, 1.0), None


@torch.no_grad()
def plot_reconstructions(
    model,
    test_loader,
    device="cpu",
    num_images=8,
    is_vae=False,
    title=None,
    save_path=None,
):
    model.eval()
    test_images, _ = next(iter(test_loader))
    test_images = test_images[:num_images].to(device)

    if is_vae:
        recon_images, _, _ = model(test_images)
    else:
        recon_images, _ = model(test_images)

    fig, axes = plt.subplots(2, num_images, figsize=(12, 3))
    for i in range(num_images):
        orig_img, cmap = _to_display_image(test_images[i])
        rec_img, _ = _to_display_image(recon_images[i])
        axes[0, i].imshow(orig_img, cmap=cmap)
        axes[0, i].axis("off")
        axes[1, i].imshow(rec_img, cmap=cmap)
        axes[1, i].axis("off")

    if title:
        plt.suptitle(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


@torch.no_grad()
def plot_vae_samples(
    vae_model,
    latent_dim=128,
    num_samples=16,
    device="cpu",
    save_path="vae_samples.png",
    title="VAE: Generated Samples from Prior z ~ N(0, I)",
):
    vae_model.eval()
    z_random = torch.randn(num_samples, latent_dim, device=device)
    generated_images = vae_model.decoder(z_random)

    cols = 8
    rows = (num_samples + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 1.5 * rows))
    for i in range(num_samples):
        r, c = divmod(i, cols)
        ax = axes[r, c] if rows > 1 else axes[c]
        img, cmap = _to_display_image(generated_images[i])
        ax.imshow(img, cmap=cmap)
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
import json
import math
import time
import numpy as np
from scipy import linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import inception_v3, Inception_V3_Weights


def _gaussian_window(window_size=11, sigma=1.5, channels=3, device="cpu"):
    coords = torch.arange(window_size, dtype=torch.float32, device=device) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_2d = g[:, None] * g[None, :]
    return window_2d.expand(channels, 1, window_size, window_size).contiguous()


def compute_ssim(img1, img2, window_size=11):
    """Compute mean SSIM between two [0, 1] image batches."""
    channels = img1.size(1)
    window = _gaussian_window(window_size, 1.5, channels, img1.device)
    pad = window_size // 2

    mu1 = F.conv2d(img1, window, padding=pad, groups=channels)
    mu2 = F.conv2d(img2, window, padding=pad, groups=channels)

    mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
    sigma1_sq = F.conv2d(img1 * img1, window, padding=pad, groups=channels) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=pad, groups=channels) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=pad, groups=channels) - mu1_mu2

    c1, c2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / (
        (mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2)
    )
    return ssim_map.mean().item()


@torch.no_grad()
def evaluate_reconstruction(model, test_loader, device="cpu", is_vae=False, max_samples=1000):
    """Evaluate MSE, PSNR (dB), and SSIM on held-out test images."""
    model.eval()
    mse_list, psnr_list, ssim_list = [], [], []
    seen = 0

    for images, _ in test_loader:
        images = images.to(device)
        recon = model(images)[0] if is_vae else model(images)[0]
        recon = torch.clamp(recon, 0.0, 1.0)

        mse = F.mse_loss(recon, images, reduction="mean").item()
        psnr = 10.0 * math.log10(1.0 / max(mse, 1e-10))
        ssim = compute_ssim(images, recon)

        mse_list.append(mse)
        psnr_list.append(psnr)
        ssim_list.append(ssim)

        seen += images.size(0)
        if seen >= max_samples:
            break

    return {
        "mse": round(float(np.mean(mse_list)), 6),
        "psnr_db": round(float(np.mean(psnr_list)), 3),
        "ssim": round(float(np.mean(ssim_list)), 4),
    }


def _load_inception_models(device):
    """Load pretrained InceptionV3 for 2048-D FID features and 1000-class IS logits."""
    weights = Inception_V3_Weights.DEFAULT
    clf = inception_v3(weights=weights).to(device).eval()
    feat_extractor = inception_v3(weights=weights).to(device).eval()
    feat_extractor.fc = nn.Identity()
    return feat_extractor, clf


def _preprocess_for_inception(imgs):
    if imgs.size(1) == 1:
        imgs = imgs.repeat(1, 3, 1, 1)
    imgs = F.interpolate(imgs, size=(299, 299), mode="bilinear", align_corners=False)
    mean = torch.tensor([0.485, 0.456, 0.406], device=imgs.device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=imgs.device).view(1, 3, 1, 1)
    return (imgs - mean) / std


@torch.no_grad()
def _extract_inception_stats(images_tensor, feat_extractor, clf, batch_size=64, device="cpu"):
    feats, probs = [], []
    for i in range(0, images_tensor.size(0), batch_size):
        batch = images_tensor[i : i + batch_size].to(device)
        inp = _preprocess_for_inception(batch)
        feats.append(feat_extractor(inp).cpu().numpy())
        logits = clf(inp)
        probs.append(F.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(feats, axis=0), np.concatenate(probs, axis=0)


def compute_fid(real_feats, gen_feats):
    mu1, sigma1 = np.mean(real_feats, axis=0), np.cov(real_feats, rowvar=False)
    mu2, sigma2 = np.mean(gen_feats, axis=0), np.cov(gen_feats, rowvar=False)
    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1 @ sigma2, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = float(diff @ diff + np.trace(sigma1 + sigma2 - 2.0 * covmean))
    return round(fid, 3)


def compute_inception_score(probs, splits=5):
    scores = []
    n = probs.shape[0]
    split_size = max(1, n // splits)
    for k in range(splits):
        part = probs[k * split_size : (k + 1) * split_size]
        if len(part) == 0:
            continue
        py = np.mean(part, axis=0, keepdims=True)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        scores.append(np.exp(np.mean(np.sum(kl, axis=1))))
    return round(float(np.mean(scores)), 3), round(float(np.std(scores)), 3)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)
    return {
        "total_params": total,
        "trainable_params": trainable,
        "model_size_mb": round(size_mb, 3),
    }


@torch.no_grad()
def benchmark_model(
    model,
    test_loader,
    latent_dim,
    device="cpu",
    is_vae=False,
    history=None,
    num_gen_samples=256,
    batch_size=64,
):
    """Run Reconstruction, Generative Quality (FID/IS), and Compute Efficiency benchmarks."""
    model.eval()
    recon_metrics = evaluate_reconstruction(model, test_loader, device=device, is_vae=is_vae)
    param_metrics = count_parameters(model)

    # Measure sampling latency & generate samples
    gen_batches = []
    if device.type == "cuda":
        torch.cuda.synchronize()
    t_start = time.time()
    for i in range(0, num_gen_samples, batch_size):
        cur_bs = min(batch_size, num_gen_samples - i)
        z = torch.randn(cur_bs, latent_dim, device=device)
        imgs = model.decoder(z) if is_vae else model.decode(z)
        gen_batches.append(torch.clamp(imgs, 0.0, 1.0).cpu())
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t_start
    gen_tensor = torch.cat(gen_batches, dim=0)

    # Collect real images for FID
    real_batches = []
    collected = 0
    for imgs, _ in test_loader:
        real_batches.append(imgs)
        collected += imgs.size(0)
        if collected >= num_gen_samples:
            break
    real_tensor = torch.cat(real_batches, dim=0)[:num_gen_samples]

    feat_extractor, clf = _load_inception_models(device)
    real_feats, _ = _extract_inception_stats(real_tensor, feat_extractor, clf, batch_size=64, device=device)
    gen_feats, gen_probs = _extract_inception_stats(gen_tensor, feat_extractor, clf, batch_size=64, device=device)

    fid = compute_fid(real_feats, gen_feats)
    is_mean, is_std = compute_inception_score(gen_probs)

    avg_epoch_time = (
        round(float(np.mean([h["epoch_time_sec"] for h in history])), 2) if history else None
    )

    return {
        "reconstruction": recon_metrics,
        "generative_quality": {
            "fid": fid,
            "inception_score_mean": is_mean,
            "inception_score_std": is_std,
            "num_eval_samples": num_gen_samples,
        },
        "compute_efficiency": {
            **param_metrics,
            "avg_epoch_time_sec": avg_epoch_time,
            "sampling_latency_ms_per_image": round((elapsed / num_gen_samples) * 1000.0, 4),
            "sampling_throughput_imgs_per_sec": round(num_gen_samples / max(elapsed, 1e-6), 2),
        },
    }


def save_benchmark_results(results, save_path="benchmark_results.json"):
    with open(save_path, "w") as f:
        json.dump(results, f, indent=2)
    return results


In [ ]:
all_training_results = {"runtime_gpu": gpu_name}
all_benchmark_results = {"runtime_gpu": gpu_name}

for ds_name in DATASETS_TO_RUN:
    print(f"\n==================== DATASET: {ds_name.upper()} ====================")
    train_loader, test_loader, cfg = get_dataloaders(dataset_name=ds_name, batch_size=BATCH_SIZE)
    in_ch, img_sz = cfg["in_channels"], cfg["image_size"]

    # 1. Convolutional Autoencoder
    ae_model = ConvAutoencoder(in_channels=in_ch, image_size=img_sz, latent_dim=LATENT_DIM).to(device)
    ae_model, ae_hist = train_autoencoder(
        ae_model, train_loader, epochs=AE_EPOCHS, lr=LR, device=device, checkpoint_path=os.path.join(RESULTS_DIR, f"ae_{ds_name}.pth")
    )
    plot_reconstructions(
        ae_model, test_loader, device=device, is_vae=False,
        title=f"{ds_name.upper()} ConvAutoencoder: Original (Top) vs Reconstructed (Bottom)",
        save_path=os.path.join(RESULTS_DIR, f"ae_{ds_name}_reconstructions.png"),
    )
    ae_bench = benchmark_model(
        ae_model, test_loader, latent_dim=LATENT_DIM, device=device, is_vae=False, history=ae_hist, num_gen_samples=256
    )

    # 2. Convolutional VAE
    vae_model = ConvVAE(in_channels=in_ch, image_size=img_sz, latent_dim=LATENT_DIM).to(device)
    vae_model, vae_hist = train_vae(
        vae_model, train_loader, epochs=VAE_EPOCHS, lr=LR, kld_weight=1.0, device=device, checkpoint_path=os.path.join(RESULTS_DIR, f"vae_{ds_name}.pth")
    )
    plot_reconstructions(
        vae_model, test_loader, device=device, is_vae=True,
        title=f"{ds_name.upper()} ConvVAE: Original (Top) vs Reconstructed (Bottom)",
        save_path=os.path.join(RESULTS_DIR, f"vae_{ds_name}_reconstructions.png"),
    )
    plot_vae_samples(
        vae_model, latent_dim=LATENT_DIM, num_samples=16, device=device,
        save_path=os.path.join(RESULTS_DIR, f"vae_{ds_name}_samples.png"),
        title=f"{ds_name.upper()} ConvVAE: Generated Samples from Prior z ~ N(0, I)",
    )
    vae_bench = benchmark_model(
        vae_model, test_loader, latent_dim=LATENT_DIM, device=device, is_vae=True, history=vae_hist, num_gen_samples=256
    )

    all_training_results[ds_name] = {"autoencoder": ae_hist, "vae": vae_hist}
    all_benchmark_results[ds_name] = {"autoencoder": ae_bench, "vae": vae_bench}

    print(f"\n[{ds_name.upper()}] Autoencoder Benchmark:", json.dumps(ae_bench, indent=2))
    print(f"[{ds_name.upper()}] VAE Benchmark:", json.dumps(vae_bench, indent=2))

save_training_results(all_training_results, save_path=os.path.join(RESULTS_DIR, "training_results.json"))
save_benchmark_results(all_benchmark_results, save_path=os.path.join(RESULTS_DIR, "benchmark_results.json"))
